# Section 6: LangChain, LlamaIndex & Agents — Hands-On

### GenAI Masterclass — From Frameworks to Autonomous Agents

---

In **Sections 1–5**, you built every piece of GenAI plumbing by hand:

- **Section 1** — How LLMs work (tokens, embeddings, attention)
- **Section 2** — Prompt engineering (zero-shot, few-shot, system prompts, JSON output)
- **Section 3** — From notebooks to apps (error handling, streaming, multi-provider)
- **Section 4** — RAG Part 1 (chunking, embeddings, ChromaDB, grounded prompts)
- **Section 5** — RAG Part 2 (hybrid search, re-ranking, evaluation)

Every helper function — `ask()`, `ask_json()`, the retry wrapper, the ChromaDB calls — you wrote yourself.

Now we **embrace frameworks**. The same patterns you wrote by hand are one-liners in LangChain and LlamaIndex. And on top of those primitives, we get something genuinely new: **agents** — LLMs that decide their own steps using tools.

This notebook walks through every concept from the Section 6 web page with running code.

---

## Setup

**Python 3.10+ recommended.** LangChain and the latest LlamaIndex use modern type syntax (`X | Y`) that fails on Python 3.9. The install cell below auto-pins LlamaIndex 0.11.x if you're still on 3.9.

We'll use:

- **OpenAI** (`gpt-4o-mini`) — main reasoning model, matches Sections 1–5
- **Groq** (`llama-3.1-8b-instant`) — fast small model, matches your Best Practices notebook
- **Tavily** — real web search for the agent
- **TechStore docs** from Section 4 — consistent teaching data for the RAG chapters

Make sure your `.env` has `OPENAI_API_KEY`, `GROQ_API_KEY`, and `TAVILY_API_KEY`.

In [24]:
# Install required libraries (run once)
import sys
import subprocess

pkgs = [
    "langchain", "langchain-openai", "langchain-groq", "langchain-community",
    "langchain-chroma", "pydantic", "langgraph", "tavily-python", "python-dotenv",
]
if sys.version_info >= (3, 10):
    pkgs += ["llama-index", "llama-index-llms-openai", "llama-index-embeddings-openai"]
else:
    print(
        f"⚠️  Python {sys.version_info.major}.{sys.version_info.minor} — "
        "pinning LlamaIndex 0.11.23 (latest LlamaIndex needs 3.10+)."
    )
    pkgs += [
        "llama-index==0.11.23",
        "llama-index-core==0.11.23",
        "llama-index-llms-openai==0.2.16",
        "llama-index-embeddings-openai==0.2.5",
    ]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print(f"✅  Installed for Python {sys.version_info.major}.{sys.version_info.minor}")

You should consider upgrading via the '/Volumes/work/teach/genai-beginner/.venv/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify keys are loaded
for key in ["OPENAI_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"]:
    status = "✅" if os.getenv(key) else "❌ MISSING"
    print(f"{status}  {key}")

✅  OPENAI_API_KEY
✅  GROQ_API_KEY
✅  TAVILY_API_KEY


In [2]:
# ── Models we'll use throughout ──
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Main reasoning model (matches Sections 1-5)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Fast small model (matches Section 2 Best Practices notebook)
small_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

print("✅ Models ready:")
print(f"   llm       = {llm.model_name} (OpenAI)")
print(f"   small_llm = {small_llm.model_name} (Groq)")

/Volumes/work/teach/genai-beginner/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Models ready:
   llm       = gpt-4o-mini (OpenAI)
   small_llm = llama-3.1-8b-instant (Groq)


---

## Chapter 3: LCEL — The Pipe Operator

### Continuing the Story...

LangChain components compose with Python's `|` operator, just like Unix shell commands. Read top-to-bottom: input → prompt → model → parser → output.

We'll start here because every LangChain example from now on uses LCEL syntax.

### 3.1 — Your First LCEL Chain

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Three components
prompt = ChatPromptTemplate.from_template("Translate '{text}' to French. Return only the translation.")
parser = StrOutputParser()

# Compose with the pipe operator
chain = prompt | llm | parser

# Run it
result = chain.invoke({"text": "Hello, world"})
print(result)

Bonjour, le monde


That's a complete chain in three lines. Compare this to what you wrote in Section 2: build a messages list, call `client.chat.completions.create`, extract `response.choices[0].message.content`, handle errors. LCEL collapses all of it.

### 3.2 — Same Chain, Different Modes (Free with LCEL)

The real win: streaming, async, and batching come for free. Same `chain` object, different methods.

In [57]:
# Mode 1: invoke (what we just did)
result = chain.invoke({"text": "Good morning"})
print(f"invoke:  {result}")

# Mode 2: batch — multiple inputs in parallel
results = chain.batch([
    {"text": "How are you?"},
    {"text": "See you tomorrow"},
    {"text": "Thank you very much"}
])
for r in results:
    print(f"batch:   {r}")

# Mode 3: stream — token-by-token output
print("stream:  ", end="", flush=True)
for chunk in chain.stream({"text": "I love programming"}):
    print(chunk, end="", flush=True)
print()

invoke:  Bonjour
batch:   Comment ça va ?
batch:   À demain.
batch:   Merci beaucoup.
stream:  J'aime programmer.


💡 **Key takeaway:** In Section 3 you wrote ~30 lines of streaming code by hand. LCEL gives it to you free, on the same chain object you already built.

### 3.3 — Swapping the Model (Provider Abstraction)

The chain doesn't care which provider sits in the middle. Swap `llm` for `small_llm` and the rest of the code is identical.

In [58]:
# Same chain, but with the small Groq model instead of OpenAI
fast_chain = prompt | small_llm | parser

result = fast_chain.invoke({"text": "Where is the library?"})
print(f"Groq Llama 3.1 8B: {result}")

Groq Llama 3.1 8B: Où est la bibliothèque ?


---

## Chapter 4: Chains in Practice

### Continuing the Story...

A chain is a pipeline. Three patterns come up over and over:

1. **Sequential refinement** — output of one step feeds the next
2. **Parallel fan-out** — multiple chains run on the same input
3. **Routing** — pick a chain based on the input

### 4.1 — Sequential: Draft → Critique → Revise

Generate a short pitch, critique it, then revise based on the critique. Output of each chain becomes input to the next.

In [61]:
from langchain_core.runnables import RunnablePassthrough

draft_prompt = ChatPromptTemplate.from_template(
    "Write a one-sentence elevator pitch for: {idea}"
)
critique_prompt = ChatPromptTemplate.from_template(
    "Critique this pitch in one sentence — what's weak about it?\n\nPitch: {draft}"
)
revise_prompt = ChatPromptTemplate.from_template(
    "Original pitch: {draft}\n"
    "Critique: {critique}\n\n"
    "Rewrite the pitch addressing the critique. One sentence only."
)

draft_chain = draft_prompt | llm | StrOutputParser()
critique_chain = critique_prompt | llm | StrOutputParser()
revise_chain = revise_prompt | llm | StrOutputParser()

# Compose sequentially. Each step's output becomes part of the next step's input.
full = (
    {"idea": RunnablePassthrough()}
    | RunnablePassthrough.assign(draft=draft_chain)
    | RunnablePassthrough.assign(critique=critique_chain)
    | revise_chain
)

result = full.invoke("a notebook app for GenAI engineers")
print(result)

"Transform your development process with our notebook app tailored for GenAI engineers, offering unique features like AI-driven code suggestions, customizable templates for model training, and integrated version control for real-time collaboration, ensuring a more efficient and innovative workflow."


In [7]:
draft_chain.invoke("i want to buy a new laptop")

'"Looking for a powerful and versatile laptop that seamlessly combines performance, portability, and style to elevate my productivity and creativity on the go!"'

### 4.2 — Parallel: Three Views in One Call

Run three chains in parallel on the same input. Total wall time ≈ slowest single chain (not sum of all three).

In [62]:
from langchain_core.runnables import RunnableParallel
import time

pros_chain = ChatPromptTemplate.from_template(
    "List 3 pros of: {topic}. Be brief."
) | llm | StrOutputParser()

cons_chain = ChatPromptTemplate.from_template(
    "List 3 cons of: {topic}. Be brief."
) | llm | StrOutputParser()

summary_chain = ChatPromptTemplate.from_template(
    "Summarize {topic} in one sentence."
) | llm | StrOutputParser()

multi_view = RunnableParallel(
    pros=pros_chain,
    cons=cons_chain,
    summary=summary_chain
)

start = time.time()
result = multi_view.invoke({"topic": "remote work"})
elapsed = time.time() - start

print(f"⏱  Wall time: {elapsed:.1f}s (all 3 ran in parallel)\n")
print("─── PROS ───");    print(result["pros"])
print("\n─── CONS ───");  print(result["cons"])
print("\n─── SUMMARY ───"); print(result["summary"])

⏱  Wall time: 2.3s (all 3 ran in parallel)

─── PROS ───
1. **Flexibility**: Employees can set their own schedules and work from anywhere, leading to better work-life balance.
2. **Cost Savings**: Reduced commuting costs and expenses related to maintaining a physical office space.
3. **Increased Productivity**: Many workers report higher productivity levels due to fewer office distractions and a personalized work environment.

─── CONS ───
1. **Isolation**: Remote work can lead to feelings of loneliness and disconnection from colleagues, impacting team cohesion and morale.

2. **Distractions**: Home environments may present more distractions, making it challenging to maintain focus and productivity.

3. **Work-Life Balance**: The blurred boundaries between work and personal life can lead to overworking and burnout.

─── SUMMARY ───
Remote work is a flexible work arrangement that allows employees to perform their job duties from locations outside of a traditional office environment, oft

### 4.3 — Routing: Pick the Right Chain Based on Input

Use a classifier chain to decide which specialist chain runs next. The LLM is doing the routing decision.

In [63]:
from langchain_core.runnables import RunnableLambda

# Three specialist chains
code_chain = ChatPromptTemplate.from_template(
    "You are a senior engineer. Answer this code question briefly: {question}"
) | llm | StrOutputParser()

math_chain = ChatPromptTemplate.from_template(
    "Solve this math problem step by step: {question}"
) | llm | StrOutputParser()

general_chain = ChatPromptTemplate.from_template(
    "Answer this question briefly: {question}"
) | llm | StrOutputParser()

# Classifier: returns one of: 'code' | 'math' | 'general'
classifier = ChatPromptTemplate.from_template(
    "Classify the question into ONE word — code, math, or general.\n\n"
    "Question: {question}\nClassification:"
) | llm | StrOutputParser()

def route(info):
    label = info["topic"].strip().lower()
    if "code" in label:
        return code_chain
    elif "math" in label:
        return math_chain
    return general_chain

router = (
    {"topic": classifier, "question": lambda x: x["question"]}
    | RunnableLambda(route)
)

# Try three different question types
for q in [
    "How do I reverse a string in Python?",
    "What is 23 * 47?",
    "What's the capital of France?"
]:
    print(f"❓ {q}")
    print(f"→  {router.invoke({'question': q})[:200]}")
    print()

❓ How do I reverse a string in Python?
→  You can reverse a string in Python using slicing. Here’s a simple example:

```python
original_string = "Hello, World!"
reversed_string = original_string[::-1]
print(reversed_string)  # Output: !dlroW

❓ What is 23 * 47?
→  To solve the multiplication problem \( 23 \times 47 \) step by step, we can use the standard algorithm for multiplication.

1. **Write the numbers vertically**:
   ```
     23
   x 47
   ```

2. **Mul

❓ What's the capital of France?
→  The capital of France is Paris.



💡 **Key takeaway:** Chains don't have to be linear. LCEL lets you compose sequential, parallel, and conditional flows using the same `|` syntax. The `RunnableParallel` and `RunnableLambda` primitives are the building blocks for everything more complex.

---

## Chapter 5: Output Parsers & Structured Output

### Reliable model outputs for real applications

### Continuing the Story...

In Section 2, you saw the pain of "please return JSON" prompts: preambles, missing fields, malformed structures.

Now we'll formalize output contracts so downstream code is stable.

### 5.1 — The Problem — Manual JSON Parsing

The fragile pattern:
- ask the model for JSON
- call `json.loads(...)`
- hope the shape is valid

This breaks when the model adds text like "Sure! Here's your JSON:" or omits fields.

In [5]:
import json

fragile_prompt = ChatPromptTemplate.from_template(
    "Extract name and age from: {text}. Return JSON with keys name and age."
)
fragile_chain = fragile_prompt | small_llm | StrOutputParser()
raw = fragile_chain.invoke({"text": "Sarah is 34 years old."})

print("=" * 70)
print("Raw model output:")
print(raw)
print("=" * 70)

try:
    parsed = json.loads(raw)
    print(f"✅ Parsed JSON: {parsed}")
except Exception as e:
    print(f"❌ json.loads failed: {e}")
    print("This is why parser abstractions matter.")

Raw model output:
You can use the following Python code to extract the name and age from the given string and return it as a JSON object:

```python
import json

def extract_info(input_string):
    # Split the string into words
    words = input_string.split()
    
    # Extract the name and age
    name = words[0]
    age = int(words[2])
    
    # Create a dictionary with the extracted information
    info = {
        "name": name,
        "age": age
    }
    
    # Return the dictionary as a JSON object
    return json.dumps(info)

# Test the function
input_string = "Sarah is 34 years old"
print(extract_info(input_string))
```

This code will output:

```json
{"name": "Sarah", "age": 34}
```

This code works by splitting the input string into words, then extracting the name (the first word) and age (the third word, which is converted to an integer). It then creates a dictionary with the extracted information and returns it as a JSON object using the `json.dumps()` function.
❌ json.

### 5.2 — JsonOutputParser — Cleaner JSON Handling

In [64]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_prompt = ChatPromptTemplate.from_template(
    "Extract name and age from: {text}\n{format_instructions}"
)
json_chain = json_prompt | llm | json_parser

json_result = json_chain.invoke({
    "text": "Sarah is 34 years old.",
    "format_instructions": json_parser.get_format_instructions(),
})

print("=" * 70)
print("✅ Parsed dict:")
print(json_result)
print(f"Type: {type(json_result)}")

✅ Parsed dict:
{'name': 'Sarah', 'age': 34}
Type: <class 'dict'>


In [65]:
print(json_parser.get_format_instructions())

Return a JSON object.


### 5.3 — PydanticOutputParser — Typed Objects

Typed parsing catches schema mistakes early and gives you real Python objects instead of ad-hoc dict juggling.

In [66]:
print(pydantic_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Person name", "title": "Name", "type": "string"}, "age": {"description": "Person age", "title": "Age", "type": "integer"}, "occupation": {"description": "Current occupation", "title": "Occupation", "type": "string"}}, "required": ["name", "age", "occupation"]}
```


In [67]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name: str = Field(description="Person name")
    age: int = Field(description="Person age")
    occupation: str = Field(description="Current occupation")

pydantic_parser = PydanticOutputParser(pydantic_object=Person)
pydantic_prompt = ChatPromptTemplate.from_template(
    "Extract a person object from: {text}\n{format_instructions}"
)
pydantic_chain = pydantic_prompt | llm | pydantic_parser

person = pydantic_chain.invoke({
    "text": "Sarah is 34 years old and works as a data analyst.",
    "format_instructions": pydantic_parser.get_format_instructions(),
})

print("=" * 70)
print("✅ Parsed Person object:")
print(person)
print(f"Type: {type(person)}")
print(f"Age next year: {person.age + 1}")

✅ Parsed Person object:
name='Sarah' age=34 occupation='data analyst'
Type: <class '__main__.Person'>
Age next year: 35


In [17]:
print(pydantic_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Person name", "title": "Name", "type": "string"}, "age": {"description": "Person age", "title": "Age", "type": "integer"}, "occupation": {"description": "Current occupation", "title": "Occupation", "type": "string"}}, "required": ["name", "age", "occupation"]}
```


### 5.4 — `with_structured_output()` — The Modern Shortcut

In [68]:
structured_llm = llm.with_structured_output(Person)

person2 = structured_llm.invoke("Extract person info: Sarah is 34 and works as a data analyst.")

print("=" * 70)
print("✅ with_structured_output result:")
print(person2)
print(f"Type: {type(person2)}")
print(f"Occupation: {person2.occupation}")

✅ with_structured_output result:
name='Sarah' age=34 occupation='data analyst'
Type: <class '__main__.Person'>
Occupation: data analyst


/Volumes/work/teach/genai-beginner/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Person(name='Sarah', age=...cupation='data analyst'), input_type=Person])
  return self.__pydantic_serializer__.to_python(


### 5.5 — Retry on Parse Failure

In [69]:
from langchain.output_parsers import OutputFixingParser

fixing_parser = OutputFixingParser.from_llm(parser=pydantic_parser, llm=small_llm)
fixing_chain = pydantic_prompt | llm | fixing_parser

fixed_person = fixing_chain.invoke({
    "text": "Sarah, 34, works in analytics.",
    "format_instructions": pydantic_parser.get_format_instructions(),
})

print("=" * 70)
print("✅ OutputFixingParser recovered structured output:")
print(fixed_person)
print(f"Type: {type(fixed_person)}")

✅ OutputFixingParser recovered structured output:
name='Sarah' age=34 occupation='analytics'
Type: <class '__main__.Person'>


💡 **Key takeaway:** Don't treat structured output as a prompt style — treat it as a contract. Parsers (and retries) turn probabilistic text generation into deterministic app interfaces.

---

## Chapter 6: Memory

### Continuing the Story...

LLM calls are stateless — the model doesn't remember the previous turn. Memory is the abstraction for managing conversation history. Four types matter in practice:

| Type | Stores | Use when |
|------|--------|----------|
| Buffer | Full history verbatim | Short chats, plenty of token budget |
| Window | Last N turns | Long chats, only recent matters |
| Summary | LLM-generated running summary | Very long chats, lossy is OK |
| Vector | Embedded turns, retrieved by relevance | Old context might matter again |

### 6.1 — A Chat Without Memory (the problem)

Watch the model fail. Each call is independent — the second message has no idea who 'I' am.

In [3]:
# No memory — just two separate LLM calls
print(llm.invoke("Hi, my name is Nij and I live in Dubai.").content)
print("---")
print(llm.invoke("What's my name and where do I live?").content)

Hi Nij! It's great to meet you. How's life in Dubai?
---
I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. I prioritize user privacy and confidentiality. How can I assist you today?


The second answer is nonsense — the model can't see the first turn. Now let's add memory.

### 6.2 — RunnableWithMessageHistory (the modern way)

The recommended pattern in modern LangChain. We provide a function that returns a history store for a given session ID.

In [7]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# Step 1: A simple chat prompt that has a slot for past messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Step 2: Plain chain
chain = chat_prompt | llm

# Step 3: Wrap with history. The store is just a dict mapping session_id → history.
store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Now both calls share the same session — the second remembers the first
config = {"configurable": {"session_id": "user-42"}}

print(chain_with_memory.invoke({"input": "Hi, my name is Nij and I live in Dubai."}, config).content)
print("---")
config = {"configurable": {"session_id": "user-43"}}
print(chain_with_memory.invoke({"input": "What's my name and where do I live?"}, config).content)

Hi Nij! It's great to meet you. How can I assist you today?
---
I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. I am designed to respect user privacy and confidentiality. How can I assist you today?


### 6.3 — Inspecting What Memory Actually Holds

The history isn't magic — it's a list of messages stored under each session ID. Let's look.

In [13]:
# Peek at what's in memory for our session
history = get_history("user-42")
print(f"Messages stored: {len(history.messages)}\n")
for i, msg in enumerate(history.messages):
    print(f"[{i}] {type(msg).__name__:<12} {msg.content[:100]}")

Messages stored: 4

[0] HumanMessage Hi, my name is Nij and I live in Dubai.
[1] AIMessage    Hi Nij! It's great to meet you. How can I assist you today?
[2] HumanMessage What's my name and where do I live?
[3] AIMessage    Your name is Nij, and you live in Dubai. How can I help you today?


### 6.4 — Window Memory: Keep Only the Last N Turns

For long chats, you don't want to send everything every time. Trim to the last N messages before passing to the model.

In [ ]:
from langchain_core.messages import trim_messages

# trim_messages keeps only the last N tokens (or messages) of history
trimmer = trim_messages(
    max_tokens=2,           # Keep only 2 messages worth
    strategy="last",
    token_counter=len,      # Use message count, not actual tokens
    include_system=True,
)

# Build a new chain that trims history before passing it in
windowed_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

windowed_chain = (
    RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x["history"]))
    | windowed_prompt
    | llm
)

windowed_with_memory = RunnableWithMessageHistory(
    windowed_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Fresh session for this demo
config2 = {"configurable": {"session_id": "windowed-user"}}

# Build up some history
windowed_with_memory.invoke({"input": "I'm allergic to peanuts."}, config2)
windowed_with_memory.invoke({"input": "I live in Dubai."}, config2)
windowed_with_memory.invoke({"input": "I'm 35 years old."}, config2)

# Now ask about something from early in the chat — likely lost due to window
result = windowed_with_memory.invoke({"input": "What am I allergic to?"}, config2)
print(result.content)

💡 **Key takeaway:** Memory is just a list of messages stored under a session ID and threaded into every prompt. Different memory types are different rules for *which* messages to include. In production, you back the store with Redis/Postgres instead of `InMemoryChatMessageHistory`.

---

## Chapter 7: Tools

### Continuing the Story...

A tool is any function the LLM can call. The model doesn't *execute* the function — it produces a structured request saying "I want to call X with these arguments." Your code runs the function and feeds the result back.

### 7.1 — Defining a Tool with @tool

In [16]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a ticker symbol like AAPL or TSLA."""
    # In real life: call a market data API
    prices = {"AAPL": 224.50, "TSLA": 248.10, "GOOGL": 178.30, "MSFT": 415.20}
    if ticker.upper() in prices:
        return f"{ticker.upper()} is trading at ${prices[ticker.upper()]}"
    return f"No price found for {ticker}"

# Inspect what the LLM will see
print(f"Tool name:        {get_stock_price.name}")
print(f"Tool description: {get_stock_price.description}")
print(f"Tool args schema: {get_stock_price.args}")

Tool name:        get_stock_price
Tool description: Get the current stock price for a ticker symbol like AAPL or TSLA.
Tool args schema: {'ticker': {'title': 'Ticker', 'type': 'string'}}


The docstring becomes the description the LLM reads. **This is what the LLM uses to decide whether to call the tool.** Write it carefully.

### 7.2 — Binding Tools to a Model

In [15]:
# Bind the tool — now the model knows it can call get_stock_price
llm_with_tools = llm.bind_tools([get_stock_price])

response = llm_with_tools.invoke("What's Apple trading at right now?")

print("Content:", response.content or "(empty — model chose to call a tool instead)")
print("\nTool calls:")
for tc in response.tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

Content: (empty — model chose to call a tool instead)

Tool calls:
  → get_stock_price({'ticker': 'AAPL'})


In [16]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_DWU3aSIURmZwCDMuaOfXPDNI', 'function': {'arguments': '{"ticker":"AAPL"}', 'name': 'get_stock_price'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 61, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_84283cb50c', 'id': 'chatcmpl-DfqyhO5v1XA7FdSXix6DLT8SRR2nX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019e2cc3-70ee-78e3-928e-682126839f46-0', tool_calls=[{'name': 'get_stock_price', 'args': {'ticker': 'AAPL'}, 'id': 'call_DWU3aSIURmZwCDMuaOfXPDNI', 'type': 'tool_call'}], usage_metadata={'input_tokens': 61, 'output_tokens': 16, 'total_tokens': 77, 'input_token_details'

Notice the model didn't *answer* — it produced a tool call. The model never executes anything. We do.

### 7.3 — The Manual Tool-Calling Loop

Before we let frameworks hide it, let's run the loop ourselves so you see exactly what's happening.

In [17]:
from langchain_core.messages import HumanMessage, ToolMessage

# Step 1: Send the user's question with tools bound
messages = [HumanMessage("What's Apple trading at, and how about Tesla?")]
response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"Step 1 — model wants to call {len(response.tool_calls)} tool(s)")
for tc in response.tool_calls:
    print(f"   → {tc['name']}({tc['args']})")

# Step 2: Execute each tool call and append results as ToolMessages
for tc in response.tool_calls:
    result = get_stock_price.invoke(tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    print(f"\nStep 2 — ran {tc['name']}: {result}")

# Step 3: Send results back to the model so it can produce the final answer
final = llm_with_tools.invoke(messages)
print(f"\nStep 3 — final answer:\n{final.content}")

Step 1 — model wants to call 2 tool(s)
   → get_stock_price({'ticker': 'AAPL'})
   → get_stock_price({'ticker': 'TSLA'})

Step 2 — ran get_stock_price: AAPL is trading at $224.5

Step 2 — ran get_stock_price: TSLA is trading at $248.1

Step 3 — final answer:
Apple (AAPL) is trading at $224.5, while Tesla (TSLA) is trading at $248.1.


💡 **Key takeaway:** That loop — call model → run tools → send results back → call model again — is what an *agent* is. An agent framework just runs this loop for you, with safety limits, retries, and tracing baked in. We'll build one in Chapter 16.

In [18]:
messages

[HumanMessage(content="What's Apple trading at, and how about Tesla?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_zKbz0B7p3hiN1816YmxiJHx4', 'function': {'arguments': '{"ticker": "AAPL"}', 'name': 'get_stock_price'}, 'type': 'function'}, {'id': 'call_7Js96oXY0nWXoLlkelxvxUW8', 'function': {'arguments': '{"ticker": "TSLA"}', 'name': 'get_stock_price'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 64, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_84283cb50c', 'id': 'chatcmpl-Dfr38mUUkrRQf6UtcgCpTD1Afq4Kd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019e2cc7-a322

---

## Chapter 8: Document Loaders & Text Splitters

### Turning raw files into retrievable chunks

### Continuing the Story...

Back in Section 4, you already used `RecursiveCharacterTextSplitter`.

Now let's learn the full LangChain ingestion idiom around it: loader → splitter → vector store. This becomes the foundation for Chapter 9 RAG chains.

### 8.1 — Document Loaders

In [8]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "../4_rag_part1/sample_docs",
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=False,
)
loaded_docs = loader.load()

print("=" * 70)
print(f"✅ Loaded docs: {len(loaded_docs)}")
print("=" * 70)
if loaded_docs:
    preview = loaded_docs[0].page_content[:100].replace("\n", " ")
    print(f"First doc preview: {preview}...")
    print(f"Metadata: {loaded_docs[0].metadata}")

✅ Loaded docs: 3
First doc preview: TechStore Employee Handbook — HR Policies Version 3.2 | Effective March 2025  Chapter 1: Paid Time O...
Metadata: {'source': '../4_rag_part1/sample_docs/employee_handbook.txt'}


### 8.2 — RecursiveCharacterTextSplitter Revisited

You've seen this splitter before in Section 4. Here we make it explicit in the LangChain pipeline and inspect chunk statistics.

In [9]:
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(loaded_docs)

by_source = Counter(d.metadata.get("source", "unknown") for d in split_docs)

print("=" * 70)
print(f"✅ Total chunks: {len(split_docs)}")
print("Chunks per source:")
for src, count in by_source.items():
    print(f"  • {src}: {count}")
print("=" * 70)
print(f"Sample chunk: {split_docs[0].page_content[:180].replace(chr(10),' ')}...")

✅ Total chunks: 31
Chunks per source:
  • ../4_rag_part1/sample_docs/employee_handbook.txt: 15
  • ../4_rag_part1/sample_docs/return_policy.txt: 9
  • ../4_rag_part1/sample_docs/product_catalog.txt: 7
Sample chunk: TechStore Employee Handbook — HR Policies Version 3.2 | Effective March 2025  Chapter 1: Paid Time Off (PTO)  Full-time employees accrue PTO at the following rates based on years o...


In [10]:
by_source

Counter({'../4_rag_part1/sample_docs/employee_handbook.txt': 15,
         '../4_rag_part1/sample_docs/return_policy.txt': 9,
         '../4_rag_part1/sample_docs/product_catalog.txt': 7})

In [11]:
split_docs

[Document(metadata={'source': '../4_rag_part1/sample_docs/employee_handbook.txt'}, page_content='TechStore Employee Handbook — HR Policies\nVersion 3.2 | Effective March 2025\n\nChapter 1: Paid Time Off (PTO)\n\nFull-time employees accrue PTO at the following rates based on years of service:\n- 0-2 years: 15 days per year (1.25 days per month)\n- 3-5 years: 20 days per year (1.67 days per month)\n- 6+ years: 25 days per year (2.08 days per month)'),
 Document(metadata={'source': '../4_rag_part1/sample_docs/employee_handbook.txt'}, page_content="PTO requests must be submitted at least 2 weeks in advance through the HR portal. Requests during peak retail periods (Black Friday week, last two weeks of December) require manager approval at least 30 days in advance. Unused PTO may be carried over to the following year, up to a maximum of 5 days. PTO balances exceeding the carry-over limit are forfeited on January 1. Upon separation from the company, accrued but unused PTO is paid out at the 

### 8.3 — Specialized Splitters

In [22]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

sample_md = """
# Return Policy
## Eligibility
### Window
Returns are accepted within 30 days.
### Condition
Product must be in original condition.
## Exceptions
Gift cards are non-refundable.
"""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")]
)
md_chunks = md_splitter.split_text(sample_md)

print("=" * 70)
print(f"✅ Markdown chunks: {len(md_chunks)}")
print("=" * 70)
for i, d in enumerate(md_chunks[:4], 1):
    print(f"Chunk {i}")
    print(f"Metadata: {d.metadata}")
    print(f"Text: {d.page_content[:90]}...")
    print("-" * 50)

✅ Markdown chunks: 3
Chunk 1
Metadata: {'h1': 'Return Policy', 'h2': 'Eligibility', 'h3': 'Window'}
Text: Returns are accepted within 30 days....
--------------------------------------------------
Chunk 2
Metadata: {'h1': 'Return Policy', 'h2': 'Eligibility', 'h3': 'Condition'}
Text: Product must be in original condition....
--------------------------------------------------
Chunk 3
Metadata: {'h1': 'Return Policy', 'h2': 'Exceptions'}
Text: Gift cards are non-refundable....
--------------------------------------------------


### 8.4 — The Three-Line Ingestion Pipeline

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 1) load  2) split  3) index
loader = DirectoryLoader("../4_rag_part1/sample_docs", glob="**/*.txt", loader_cls=TextLoader)
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
lc_vectorstore = Chroma.from_documents(chunks, embedding=OpenAIEmbeddings())

print("=" * 70)
print("✅ Ingestion complete: loader → splitter → Chroma")
print(f"Docs: {len(docs)} | Chunks: {len(chunks)}")
print("This store is reused in Chapter 9.")

✅ Ingestion complete: loader → splitter → Chroma
Docs: 3 | Chunks: 31
This store is reused in Chapter 9.


💡 **Key takeaway:** Document loaders standardize input, splitters shape retrieval quality, and vector stores make semantic retrieval fast. Most RAG quality issues start in this ingestion pipeline — not at generation time.

---

## Chapter 9: Retrievers & RAG in LangChain

### The LangChain idiom for everything you built in Sections 4–5

### Continuing the Story...

In Section 4, you built RAG manually from primitives. That was essential for intuition.

Now let's close the loop with LangChain's composed idiom: `as_retriever` + `create_stuff_documents_chain` + `create_retrieval_chain`.

### 9.1 — Building a Retriever from a Vector Store

In [ ]:
retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 4})

hits = retriever.invoke("What is TechStore's return policy?")
print("=" * 70)
print(f"✅ Retrieved docs: {len(hits)}")
print("=" * 70)
for i, d in enumerate(hits, 1):
    snippet = d.page_content[:160].replace("\n", " ")
    print(f"{i}. {snippet}...")

✅ Retrieved docs: 4
1. TechStore Return Policy Last Updated: January 2025  1. Standard Return Policy Standard items may be returned within 30 days of the original purchase date. A val...
2. 9. Exceptions The store manager has discretion to approve returns outside of standard policy for customer satisfaction purposes. Manager-approved exceptions are...
3. 2. Electronics Return Policy All electronics (laptops, tablets, phones, monitors, peripherals) have a shortened return window of 15 days from the date of purcha...
4. 5. Holiday Return Policy Purchases made between November 15 and December 31 of any calendar year are eligible for an extended return window through January 31 o...


### 9.2 — The Stuff Documents Pattern

`stuff` means: place all retrieved chunks into a single prompt context block, then ask the model to answer from that context only.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.combine_documents import create_stuff_documents_chain

stuff_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based ONLY on the context. If the answer is not present, say you don't know.\n\nContext:\n{context}"),
    ("human", "{input}")
])

combine_docs_chain = create_stuff_documents_chain(llm, stuff_prompt)
print("✅ create_stuff_documents_chain ready")

✅ create_stuff_documents_chain ready


### 9.3 — Full RAG Chain

In [15]:
from langchain.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(retriever, combine_docs_chain)
rag_result = rag_chain.invoke({"input": "What's our return policy?"})

print("=" * 70)
print("🤖 Answer:")
print(rag_result["answer"])
print("=" * 70)
print("Retrieved context summary:")
for i, d in enumerate(rag_result["context"], 1):
    print(f"{i}. {d.page_content[:120].replace(chr(10), ' ')}...")

🤖 Answer:
The return policy allows standard items to be returned within 30 days of the original purchase date with a valid receipt or proof of purchase. Items must be in unused, resalable condition with all original tags and packaging intact. Opened items may incur a 15% restocking fee at the manager's discretion. 

For electronics, the return window is shortened to 15 days from the date of purchase, and they must include all original packaging, accessories, cables, and documentation. Opened software or activated digital products bundled with electronics are non-refundable, and electronics showing physical damage are not eligible for return.

There are exceptions where the store manager can approve returns outside of the standard policy for customer satisfaction, limited to one per customer per calendar year. Items from third-party sellers follow their individual return policies, and clearance items marked as "Final Sale" are non-returnable and non-refundable.

Additionally, purchases 

In [16]:
rag_result

{'input': "What's our return policy?",
 'context': [Document(id='9206e15c-53d8-43ca-9482-7ddb71bd42a1', metadata={'source': '../4_rag_part1/sample_docs/return_policy.txt'}, page_content="TechStore Return Policy\nLast Updated: January 2025\n\n1. Standard Return Policy\nStandard items may be returned within 30 days of the original purchase date. A valid receipt or proof of purchase is required for \n\n\nall returns. Items must be in unused, resalable condition with all original tags and packaging intact. Opened items may be subject to a 15% restocking fee at the manager's discretion."),
  Document(id='e2131a39-c368-465a-9ca1-f5b6d6ac8560', metadata={'source': '../4_rag_part1/sample_docs/return_policy.txt'}, page_content='2. Electronics Return Policy\nAll electronics (laptops, tablets, phones, monitors, peripherals) have a shortened return window of 15 days from the date of purchase. Electronics must include all original packaging, accessories, cables, and documentation. Opened software o

### 9.4 — History-Aware Retrieval

Multi-turn chat introduces ambiguity: users say things like "tell me more about that."

History-aware retrieval rewrites the follow-up into a standalone query before retrieval.

In [17]:
from langchain.chains import create_history_aware_retriever
from langchain_core.messages import HumanMessage, AIMessage

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the latest user question into a standalone query using chat history."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

history_aware = create_history_aware_retriever(llm, retriever, rewrite_prompt)

chat_history = [
    HumanMessage(content="What is TechStore's return policy?"),
    AIMessage(content="Returns are accepted within 30 days in original condition."),
]

follow_up_docs = history_aware.invoke({
    "chat_history": chat_history,
    "input": "Tell me more about that"
})

print("=" * 70)
print("✅ History-aware retriever resolved the ambiguous follow-up")
print("=" * 70)
for i, d in enumerate(follow_up_docs, 1):
    print(f"{i}. {d.page_content[:120].replace(chr(10), ' ')}...")

✅ History-aware retriever resolved the ambiguous follow-up
1. TechStore Return Policy Last Updated: January 2025  1. Standard Return Policy Standard items may be returned within 30 d...
2. 2. Electronics Return Policy All electronics (laptops, tablets, phones, monitors, peripherals) have a shortened return w...
3. 9. Exceptions The store manager has discretion to approve returns outside of standard policy for customer satisfaction p...
4. 5. Holiday Return Policy Purchases made between November 15 and December 31 of any calendar year are eligible for an ext...


### 9.5 — Comparing to Section 4

Section 4's manual pipeline took roughly ~80 lines so you could see every moving part.

This chapter recreated the same behavior in roughly ~15 lines of composable primitives:
- retriever from vector store
- context stuffing chain
- retrieval chain composition
- optional history-aware query rewrite

The abstraction is compact, but now you understand what each layer is doing.

💡 **Key takeaway:** LangChain RAG is a small composition of reusable primitives. You get less boilerplate, easier testing, and clear extension points (rerankers, history-aware rewrite, guardrails) without rewriting the full pipeline.

---

## Chapter 10: Callbacks & LangSmith

### Continuing the Story...

LangSmith is LangChain's hosted observability platform. With two environment variables, every chain run shows up in a UI with full inputs, outputs, latency, cost, and intermediate steps.

**You don't need a paid plan to follow along** — LangSmith has a free tier that covers personal use comfortably.

### 10.1 — Enabling Tracing

Set these in your `.env`:

```
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__your_key_here
LANGCHAIN_PROJECT=genai-decoded-section-6
```

Once enabled, **every** `chain.invoke()` from this notebook gets logged to LangSmith automatically — no other code changes needed.

In [ ]:
# Check whether tracing is on
tracing_on = os.getenv("LANGCHAIN_TRACING_V2") == "true"
project = os.getenv("LANGCHAIN_PROJECT", "(not set)")

if tracing_on:
    print(f"✅ LangSmith tracing is ON")
    print(f"   Project: {project}")
    print(f"   Visit https://smith.langchain.com to see your traces")
else:
    print("ℹ️  LangSmith tracing is OFF.")
    print("   Set LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY in .env to enable.")
    print("   Then re-run any chain in this notebook to see it appear in LangSmith.")

✅ LangSmith tracing is ON
   Project: genai-decoded-section-6
   Visit https://smith.langchain.com to see your traces


💡 **Key takeaway:** Production rule — turn on tracing from day one. The cost of *not* having traces when something breaks in production is far higher than the (very small) cost of running them.

---

## Chapter 11: LlamaIndex Data Connectors

### Switching Frameworks: Welcome to LlamaIndex

LangChain shines at orchestration. LlamaIndex shines at **getting your data in**. The LlamaHub registry has 300+ connectors, all returning the same `Document` shape.

We'll use the TechStore docs from Section 4 as our consistent teaching data.

### 11.1 — Setting Up Sample Documents

Reusing TechStore docs from Section 4. If they're not present in the expected location, we'll create minimal versions inline so the notebook is self-contained.

In [3]:
from pathlib import Path

# Path used in Section 4
techstore_dir = Path("../4_rag_part1/sample_docs")
local_dir = Path("./sample_docs")

if techstore_dir.exists():
    docs_dir = techstore_dir
    print(f"✅ Using TechStore docs from Section 4: {docs_dir.resolve()}")
else:
    # Fallback: create minimal copies so the notebook still works standalone
    local_dir.mkdir(exist_ok=True)
    (local_dir / "return_policy.txt").write_text(
        "TechStore Return Policy\n\n"
        "Customers may return any product within 30 days of purchase for a full refund. "
        "Returns must include the original packaging and proof of purchase. "
        "Opened software is non-refundable. "
        "Damaged items can be exchanged but not refunded."
    )
    (local_dir / "employee_handbook.txt").write_text(
        "TechStore Employee Handbook\n\n"
        "All employees receive 20 days of paid leave per year. "
        "Remote work is permitted up to 2 days per week with manager approval. "
        "Performance reviews happen quarterly. "
        "Health insurance includes dental and vision coverage."
    )
    (local_dir / "product_catalog.txt").write_text(
        "TechStore Product Catalog\n\n"
        "Laptops: ProBook 15 ($1,299), UltraSlim 13 ($999), GamingMax 17 ($1,899). "
        "All laptops come with a 2-year warranty. "
        "Accessories: wireless mouse ($29), USB-C hub ($49), laptop stand ($79)."
    )
    docs_dir = local_dir
    print(f"ℹ️  TechStore docs not found — created fallback in: {docs_dir.resolve()}")

print(f"\nFiles available:")
for f in sorted(docs_dir.iterdir()):
    print(f"   {f.name}")

✅ Using TechStore docs from Section 4: /Volumes/work/teach/genai-beginner/coaching_material/training_materials/4_rag_part1/sample_docs

Files available:
   employee_handbook.txt
   product_catalog.txt
   return_policy.txt
   sample_report.pdf
   sample_report_V1.pdf
   scanned_page.png


### 11.2 — SimpleDirectoryReader: One Line, Many Formats

`SimpleDirectoryReader` handles `.txt`, `.pdf`, `.docx`, `.md`, `.csv`, images, and more — automatically picking the right parser per extension.

In [10]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(str(docs_dir)).load_data()

print(f"Loaded {len(documents)} documents\n")
for doc in documents:
    print(f"📄 {doc.metadata.get('file_name', 'unknown')}")
    print(f"   Length: {len(doc.text)} chars")
    print(f"   Preview: {doc.text[:120]}...")
    print()

incorrect startxref pointer(1)
parsing for Object Streams


Loaded 7 documents

📄 employee_handbook.txt
   Length: 3780 chars
   Preview: TechStore Employee Handbook — HR Policies
Version 3.2 | Effective March 2025

Chapter 1: Paid Time Off (PTO)

Full-time ...

📄 product_catalog.txt
   Length: 2552 chars
   Preview: TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our flagship laptop. 15.6-inch 4K OLED displa...

📄 return_policy.txt
   Length: 3570 chars
   Preview: TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
Standard items may be returned within 30 d...

📄 sample_report.pdf
   Length: 476 chars
   Preview: TechStore Q1 2025 Sales Report
Total revenue for Q1 2025 was $4.2 million, representing a 15% increase over Q1 2024. The...

📄 sample_report_V1.pdf
   Length: 1094 chars
   Preview: TechStore Q1 2025 Sales Report
Total revenue for Q1 2025 was $4.2 million, representing a 15% increase over Q1 2024. The...

📄 sample_report_V1.pdf
   Length: 589 chars
   Preview: 1. Launch the new Pro

### 11.3 — Custom Connector: Web Page

Different connector, same `Document` output. The downstream pipeline doesn't care where data came from.

In [9]:
# Note: only run this cell if you have internet access
from llama_index.readers.web import SimpleWebPageReader

# Same uniform Document interface, different source
web_docs = SimpleWebPageReader().load_data([
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
])

print(f"Loaded {len(web_docs)} web document(s)")
print(f"First 200 chars:\n{web_docs[0].text[:200]}...")

Loaded 1 web document(s)
First 200 chars:
Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.
...


💡 **Key takeaway:** All 300+ LlamaIndex connectors return the same `Document` shape. Load once with the right connector — the rest of your pipeline (chunking, indexing, querying) stays identical regardless of source.

---

## Chapter 12: Indices — Beyond Just Vector Search

### Continuing the Story...

In Sections 4–5, "the index" meant a vector store. LlamaIndex generalizes: an **index** is any data structure built over your documents to make retrieval efficient.

We'll demo the two most common types — `VectorStoreIndex` (the default) and `SummaryIndex` (for exhaustive coverage).

### 12.1 — VectorStoreIndex: Three Lines to Working RAG

In [8]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI as LIOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LlamaIndex to use the same models as our LangChain code
Settings.llm = LIOpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# The whole RAG pipeline in three lines
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = query_engine.query("What's our return policy?")

print(response)

The return policy includes several key points:

1. **Standard Return Policy**: Items can be returned within 30 days with a valid receipt, in unused and resalable condition. Opened items may incur a 15% restocking fee.

2. **Electronics Return Policy**: Electronics must be returned within 15 days, including all original packaging and accessories. Opened software or activated digital products are non-refundable.

3. **Software and Digital Products**: Software is non-refundable once the activation code is used. Unopened software can be returned within 30 days, while digital download codes are non-refundable.

4. **Gift Cards**: Gift cards are non-refundable and cannot be exchanged. Lost cards require proof of purchase for replacement.

5. **Holiday Return Policy**: Purchases made between November 15 and December 31 can be returned until January 31 of the following year, but electronics follow the standard 15-day policy.

6. **Damaged Items**: Damaged items must be reported within 48 hours

Compare this to your Section 4 code: you wrote ~80 lines for chunking, embedding, ChromaDB setup, retrieval, and prompt assembly. LlamaIndex does it in three lines with sensible defaults.

### 12.2 — Inspecting What Was Retrieved

The response includes the source nodes that grounded the answer.

In [11]:
# Same query, but inspect the retrieved chunks
response = query_engine.query("Can I return opened software?")

print(f"Answer: {response}\n")
print(f"Retrieved {len(response.source_nodes)} chunks:\n")
for i, node in enumerate(response.source_nodes):
    print(f"[{i}] Score: {node.score:.3f}")
    print(f"    Source: {node.metadata.get('file_name', 'unknown')}")
    print(f"    Text: {node.text[:150]}...")
    print()

Answer: Opened software is non-refundable once the activation code or license key has been used or revealed. However, unopened, shrink-wrapped software may be returned within 30 days.

Retrieved 2 chunks:

[0] Score: 0.444
    Source: return_policy.txt
    Text: TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
Standard items may be returned within 30 days of the original purchase d...

[1] Score: 0.201
    Source: product_catalog.txt
    Text: TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our flagship laptop. 15.6-inch 4K OLED display, Intel i9-14900H processor, ...



### 12.3 — SummaryIndex: Read Everything, Miss Nothing

When the question is "summarize all of X" rather than "find specific Y", `SummaryIndex` reads through every chunk instead of relying on top-K retrieval.

In [12]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex.from_documents(documents)
summary_engine = summary_index.as_query_engine()

# A question where exhaustive coverage matters
response = summary_engine.query(
    "Summarize all the policies and benefits TechStore offers, across all documents."
)
print(response)

TechStore provides a range of policies and benefits for its employees, which include:

1. **Paid Time Off (PTO)**: Full-time employees accrue PTO based on their years of service, with 15 days for 0-2 years, 20 days for 3-5 years, and 25 days for 6+ years. Requests must be submitted in advance, and unused PTO can be carried over up to 5 days.

2. **Remote Work Policy**: Corporate and technology employees can work remotely up to 3 days a week with manager approval, while retail employees are not eligible. Remote workers must be available during core hours and maintain a secure workspace. A one-time home office stipend of $500 is provided.

3. **Performance Reviews**: Conducted twice a year, these reviews use a 5-point scale and determine merit increases and bonus eligibility. Employees rated 2 or below may enter a Performance Improvement Plan.

4. **Benefits**: Health insurance begins after 30 days of employment, with three plan options. The company matches 401(k) contributions up to 4% 

💡 **Key takeaway:** `VectorStoreIndex` for "find the relevant bit" — fast, top-K. `SummaryIndex` for "read everything and tell me" — slower but exhaustive. Pick the index that matches the question shape.

---

## Chapter 13: Query Engines & Routers

### Continuing the Story...

A `RouterQueryEngine` uses an LLM to pick *which* index to query based on the question. This is how production apps with multiple data domains work.

### 13.1 — Building a Router Over Two Indices

We'll build separate indices for the return policy and the employee handbook, then let a router LLM decide which to use per question.

In [13]:
# Build two specialist indices, one per document
return_doc = [d for d in documents if "return" in d.metadata.get("file_name", "").lower()]
hr_doc     = [d for d in documents if "employee" in d.metadata.get("file_name", "").lower()]

return_index = VectorStoreIndex.from_documents(return_doc) if return_doc else None
hr_index     = VectorStoreIndex.from_documents(hr_doc)     if hr_doc else None

print(f"return_index built: {return_index is not None}")
print(f"hr_index built:     {hr_index is not None}")

return_index built: True
hr_index built:     True


In [14]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# Wrap each index as a tool with a clear description
# (the description is what the router LLM reads to choose)
tools = []
if return_index:
    tools.append(QueryEngineTool.from_defaults(
        query_engine=return_index.as_query_engine(),
        description="Use for questions about returns, refunds, exchanges, and product purchase policies."
    ))
if hr_index:
    tools.append(QueryEngineTool.from_defaults(
        query_engine=hr_index.as_query_engine(),
        description="Use for questions about employee benefits, leave, remote work, and HR policies."
    ))

router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=tools,
    verbose=True,  # Show which tool the router picks
)

# Two questions, each routes to a different tool
print("\n=== Question 1 ===")
print(router.query("How many vacation days do employees get?"))

print("\n=== Question 2 ===")
print(router.query("Can I return opened software?"))


=== Question 1 ===
Selecting query engine 1: The question pertains to employee benefits and leave policies..
Employees accrue vacation days, referred to as Paid Time Off (PTO), based on their years of service as follows:
- 0-2 years: 15 days per year
- 3-5 years: 20 days per year
- 6+ years: 25 days per year

=== Question 2 ===
Selecting query engine 0: The question pertains to returns and product purchase policies..
Opened software may only be returned if it is unopened and shrink-wrapped. Once the activation code or license key has been used or revealed, the software is non-refundable.


💡 **Key takeaway:** A router is an LLM picking between options based on descriptions — the same primitive that powers full agents in the next chapter. Routers are agents with one decision; agents are routers with many.

---

## Chapter 15: The ReAct Loop

### Welcome to Agents

We've reached the centerpiece. An **agent** is an LLM in a loop with tools, deciding its own steps. The dominant pattern is **ReAct** — at each step the model produces a *Thought*, an *Action* (tool call), and processes the *Observation* (tool result), until it decides it's done.

### 15.1 — A Tiny Agent with One Tool

Let's start small — one tool, one query, full visibility into what the agent is doing.

In [17]:
from langgraph.prebuilt import create_react_agent

# Reuse our stock price tool from Chapter 7
agent = create_react_agent(llm, [get_stock_price])

result = agent.invoke({
    "messages": [("user", "What's Apple trading at?")]
})

# Print every message in the trace so we see the loop
for msg in result["messages"]:
    msg_type = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"[{msg_type}] tool_calls = {[tc['name'] for tc in msg.tool_calls]}")
    else:
        content = msg.content[:200] if msg.content else "(empty)"
        print(f"[{msg_type}] {content}")
    print()

[HumanMessage] What's Apple trading at?

[AIMessage] tool_calls = ['get_stock_price']

[ToolMessage] AAPL is trading at $224.5

[AIMessage] Apple (AAPL) is currently trading at $224.50.



You should see four messages: HumanMessage (user query) → AIMessage with tool_calls → ToolMessage (result) → AIMessage (final answer). That's one ReAct iteration.

### 15.2 — Watching a Multi-Step Trace

Now a question that needs two tool calls. Watch the agent chain them.

In [22]:
# Add a second tool: percent change calculator
@tool
def percent_change(start: float, end: float) -> str:
    """Calculate the percent change from start to end. Returns a string like '+12.3%'."""
    pct = (end - start) / start * 100
    sign = "+" if pct >= 0 else ""
    return f"{sign}{pct:.2f}%"

agent_v2 = create_react_agent(llm, [get_stock_price, percent_change])

result = agent_v2.invoke({
    "messages": [("user",
        "Apple was at $180 a year ago. What's the percent change to today's price?")]
})

# print(result["messages"])
# Pretty-print the trace
for i, msg in enumerate(result["messages"]):
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{i}] {label}  →  {tc['name']}({tc['args']})")
    elif label == "ToolMessage":
        print(f"[{i}] {label}  ←  {msg.content}")
    else:
        content = msg.content[:200] if msg.content else "(empty)"
        print(f"[{i}] {label}: {content}")

[0] HumanMessage: Apple was at $180 a year ago. What's the percent change to today's price?
[1] AIMessage  →  get_stock_price({'ticker': 'AAPL'})
[2] ToolMessage  ←  AAPL is trading at $224.5
[3] AIMessage  →  percent_change({'start': 180, 'end': 224.5})
[4] ToolMessage  ←  +24.72%
[5] AIMessage: The percent change in Apple's stock price from $180 a year ago to today's price of $224.5 is +24.72%.


💡 **Key takeaway:** The agent figured out it needed `get_stock_price` first (for today's price), *then* `percent_change` (using last year's price + today's). You wrote zero orchestration code — the LLM ordered the calls.

---

## Chapter 16: Building a Real Agent

### The Capstone

Three tools — web search (real, via Tavily), calculator, exchange rates — wired into one agent. Then we ask a question that needs all three.

### 16.1 — Defining the Three Tools

In [23]:
from tavily import TavilyClient
import requests

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> str:
    """Search the web for current events, news, prices, sports scores, or any
    information that may have changed recently. Use this for anything where
    fresh real-world data matters. Do NOT use for general knowledge or pure math."""
    results = tavily.search(query, max_results=3)
    snippets = [r["content"] for r in results.get("results", [])]
    return "\n\n".join(snippets) if snippets else "No results."

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression in Python syntax.
    Example: calculate('(224.50 - 180.30) / 180.30 * 100')"""
    try:
        # Sandboxed eval — no builtins, no globals
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    """Get the current exchange rate between two currencies.
    Currencies are 3-letter codes like USD, EUR, AED, INR."""
    r = requests.get(f"https://api.exchangerate-api.com/v4/latest/{from_currency.upper()}")
    rates = r.json().get("rates", {})
    rate = rates.get(to_currency.upper())
    if rate is None:
        return f"Could not find rate for {from_currency} → {to_currency}"
    return f"1 {from_currency.upper()} = {rate} {to_currency.upper()}"

print("✅ Three tools defined")
for t in [web_search, calculate, get_exchange_rate]:
    print(f"   • {t.name}: {t.description.split(chr(10))[0]}")

✅ Three tools defined
   • web_search: Search the web for current events, news, prices, sports scores, or any
   • calculate: Evaluate a mathematical expression in Python syntax.
   • get_exchange_rate: Get the current exchange rate between two currencies.


In [25]:
tools = [web_search, calculate, get_exchange_rate]
tools

[StructuredTool(name='web_search', description='Search the web for current events, news, prices, sports scores, or any\n    information that may have changed recently. Use this for anything where\n    fresh real-world data matters. Do NOT use for general knowledge or pure math.', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search at 0x175b73550>),
 StructuredTool(name='calculate', description="Evaluate a mathematical expression in Python syntax.\n    Example: calculate('(224.50 - 180.30) / 180.30 * 100')", args_schema=<class 'langchain_core.utils.pydantic.calculate'>, func=<function calculate at 0x175bdeaf0>),
 StructuredTool(name='get_exchange_rate', description='Get the current exchange rate between two currencies.\n    Currencies are 3-letter codes like USD, EUR, AED, INR.', args_schema=<class 'langchain_core.utils.pydantic.get_exchange_rate'>, func=<function get_exchange_rate at 0x175b5b670>)]

### 16.2 — Wiring the Agent and Running It

One `create_react_agent` call. The LangGraph runtime handles the loop, retries, parallel calls, and message threading.

We'll build the same behavior manually in Chapter 17 to see the graph under the hood.

In [26]:
tools = [web_search, calculate, get_exchange_rate]
real_agent = create_react_agent(llm, tools)

question = (
    "What's Apple's current stock price in AED, "
    "and how much higher (in dollars) is it than $200?"
)

result = real_agent.invoke({"messages": [("user", question)]})

# Print the full trace
print("=" * 70)
print(f"USER: {question}")
print("=" * 70)
for msg in result["messages"][1:]:  # skip the user message we already printed
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            args_short = str(tc['args'])[:120]
            print(f"\n→ TOOL CALL: {tc['name']}({args_short})")
    elif label == "ToolMessage":
        content = msg.content[:300]
        print(f"← TOOL RESULT: {content}")
    elif msg.content:
        print(f"\n🤖 FINAL ANSWER:\n{msg.content}")

USER: What's Apple's current stock price in AED, and how much higher (in dollars) is it than $200?

→ TOOL CALL: web_search({'query': 'Apple current stock price'})

→ TOOL CALL: get_exchange_rate({'from_currency': 'USD', 'to_currency': 'AED'})
← TOOL RESULT: The current price of AAPL is 313.67 USD — it has increased by 0.31% in the past 24 hours. Watch Apple Inc stock price performance more closely on the chart. 00

Apple(AAPL) trades at $314.14. Shares are currently priced at $314.14, which is +1.2% above the low and -0.3% below the high. a volume of 2
← TOOL RESULT: 1 USD = 3.67 AED

→ TOOL CALL: calculate({'expression': '313.67 - 200'})
← TOOL RESULT: 113.67000000000002

🤖 FINAL ANSWER:
Apple's current stock price is approximately **$313.67**. This is **$113.67** higher than $200.

Additionally, the current exchange rate is **1 USD = 3.67 AED**. If you need the stock price in AED, let me know!


### 16.3 — Streaming the Agent's Reasoning

For real apps, you want to see the agent think in real time. `astream_events` gives you a token-by-token stream of every step.

In [27]:
# Stream version — see steps as they happen
print("Streaming agent steps:\n")

for chunk in real_agent.stream(
    {"messages": [("user", "What's the current price of Tesla in EUR?")]},
    stream_mode="values"
):
    last_msg = chunk["messages"][-1]
    label = type(last_msg).__name__
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"→  {tc['name']}({tc['args']})")
    elif label == "ToolMessage":
        print(f"←  {last_msg.content[:200]}")
    elif last_msg.content and label == "AIMessage":
        print(f"🤖  {last_msg.content[:300]}")

Streaming agent steps:

→  web_search({'query': 'current Tesla stock price in EUR'})
←  EUR Tesla, Inc. (TL0.DE) 348.65 -12.15 .55% YTD -11.40% Range 348.10 - 366.50 52 Week Range 246.25 - 424.00 Volume 22,182

Xetra (EUR)· Market open 360.30 0.35+0.10% Delayed price as of 6:13 AM EDT 06
🤖  The current price of Tesla in EUR is approximately €360.30.


### 16.4 — Setting Safety Limits

Real agents need guardrails. The two most important: max iterations (prevents infinite loops) and recursion limit (prevents runaway recursion).

In [28]:
from langgraph.errors import GraphRecursionError

# Configure with a tight recursion limit
try:
    result = real_agent.invoke(
        {"messages": [("user", "What's the price of Microsoft stock?")]},
        config={"recursion_limit": 10}  # Hard ceiling — stops if exceeded
    )
    print("✅ Completed within limits")
    print(result["messages"][-1].content[:300])
except GraphRecursionError as e:
    print(f"❌ Hit recursion limit: {e}")

✅ Completed within limits
I couldn't retrieve the current price of Microsoft stock directly. However, you can check the latest price on financial news websites or stock market platforms like Yahoo Finance, Google Finance, or Bloomberg. Would you like me to help you with anything else?


💡 **Key takeaway:** You wrote three small functions and one `create_react_agent` line. The LLM picked the tools, ordered the calls, threaded the results, and produced the final answer. That's the leverage frameworks give you — and why we spent five sections building primitives by hand first, so you'd appreciate it.

---

## Chapter 17: LangGraph — State Machines for Agents

### When `create_react_agent` isn't enough

### Continuing the Story...

`create_react_agent` feels like a one-liner magic trick. Under the hood, it's a graph with state transitions.

In this chapter, we'll peel that abstraction open and build the same loop ourselves so you understand what production teams customize (HITL, checkpoints, branching, persistence).

### 17.1 — The Three Primitives — State, Nodes, Edges

- **State**: the shared data each step reads/writes (messages, tool outputs, user metadata).
- **Nodes**: functions that do work (call model, call tools, run validators).
- **Edges**: routing rules deciding which node runs next.

This is the full agent mental model. `create_react_agent` prebuilds this graph for the common ReAct case.

### 17.2 — A Minimal LangGraph Agent from Scratch

In [29]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

langgraph_tools = [web_search, calculate, get_exchange_rate]
tool_node = ToolNode(langgraph_tools)
llm_with_tools = llm.bind_tools(langgraph_tools)


def call_model(state: AgentState) -> AgentState:
    ai_msg = llm_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def call_tools(state: AgentState) -> AgentState:
    tool_result = tool_node.invoke(state)
    return {"messages": tool_result["messages"]}


def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    return "tools" if getattr(last_message, "tool_calls", None) else "end"

builder = StateGraph(AgentState)
builder.add_node("model", call_model)
builder.add_node("tools", call_tools)
builder.set_entry_point("model")
builder.add_conditional_edges("model", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "model")
manual_graph_agent = builder.compile()

q = "What's Apple's current stock price in AED, and how much higher (in dollars) is it than $200?"
result = manual_graph_agent.invoke({"messages": [HumanMessage(content=q)]})

print("=" * 70)
print(f"USER: {q}")
print("=" * 70)
for msg in result["messages"]:
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"→ TOOL CALL: {tc['name']}({tc['args']})")
    elif label == "ToolMessage":
        print(f"← TOOL RESULT: {str(msg.content)[:250]}")
    elif label == "AIMessage" and msg.content:
        print(f"🤖 FINAL ANSWER: {str(msg.content)[:350]}")

USER: What's Apple's current stock price in AED, and how much higher (in dollars) is it than $200?
→ TOOL CALL: web_search({'query': 'Apple current stock price'})
→ TOOL CALL: get_exchange_rate({'from_currency': 'USD', 'to_currency': 'AED'})
← TOOL RESULT: Error: ConnectionError(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')))
 Please fix your mistakes.
← TOOL RESULT: 1 USD = 3.67 AED
→ TOOL CALL: web_search({'query': 'Apple current stock price'})
← TOOL RESULT: Stock Quote: NASDAQ: AAPL ; Day's Open313.23 ; Closing Price311.23 ; Volume44.9 ; Intraday High313.54 ; Intraday Low309.65.

Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with your stock trading 
→ TOOL CALL: calculate({'expression': '314.23 - 200'})
← TOOL RESULT: 114.23000000000002
🤖 FINAL ANSWER: Apple's current stock price is $314.23. It is $114.23 higher than $200. 

Additionally, the exchange rate is 1 USD = 3.67 AED. If you 

### 17.3 — Why Bother — Human-in-the-Loop Checkpoints

This matters when tools can do risky actions (payments, deleting records, sending external messages).

Pattern:
1. Interrupt right before tool execution.
2. Show pending tool call to a human reviewer.
3. Approve/reject and resume from exactly that state.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpoint_graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["tools"]
)

config = {"configurable": {"thread_id": "hitl-demo"}}

print("=" * 70)
print("Step 1: Run until interrupt before tool execution")
print("=" * 70)
first = checkpoint_graph.invoke(
    {"messages": [HumanMessage(content="What's Tesla stock in EUR right now?")]},
    config=config
)

state = checkpoint_graph.get_state(config)
print(f"✅ Interrupted at node: {state.next}")
print("Pending tool call preview:")
last = state.values["messages"][-1]
for tc in getattr(last, "tool_calls", []):
    print(f"   • {tc['name']}({tc['args']})")

print("\n" + "=" * 70)
print("Step 2: Human approves and resumes")
print("=" * 70)
approved = checkpoint_graph.invoke(None, config=config)
print(approved["messages"][-1].content[:350])

### 17.4 — Streaming Graph Execution

In [ ]:
print("Streaming node-by-node execution:\n")

for event in manual_graph_agent.stream(
    {"messages": [HumanMessage(content="What's Nvidia price in USD and convert to AED?")]}
):
    for node_name, payload in event.items():
        if "messages" not in payload:
            continue
        last_msg = payload["messages"][-1]
        label = type(last_msg).__name__
        print("=" * 70)
        print(f"Node: {node_name} | Message type: {label}")
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f"→ {tc['name']}({tc['args']})")
        else:
            print(str(last_msg.content)[:250])

### 17.5 — When to Use LangGraph vs `create_react_agent`

Use **`create_react_agent`** when you want speed and default good behavior.

Use **LangGraph directly** when you need:
- human approval checkpoints before specific tools
- custom planning loops and branch logic
- parallel branches and merges
- durable execution with checkpoints/persistence
- multi-agent orchestration in one graph

💡 **Key takeaway:** `create_react_agent` is the fast path; LangGraph is the control path. They are not competing tools — one is the convenience wrapper, the other is the runtime you reach for when product requirements get real.

---

## Wrap-Up: What You've Built

In this notebook you now have running examples of:

- **LCEL pipelines** — `prompt | llm | parser`, streaming, batching, async
- **Output parsers** — `StrOutputParser`, `JsonOutputParser`, `PydanticOutputParser`, and `with_structured_output()`
- **Composition patterns** — sequential, parallel, and conditional routing
- **Memory** — `RunnableWithMessageHistory` with sessions + trimming
- **Tools and agents** — manual tool loop, `create_react_agent`, and multi-tool orchestration
- **Document loaders + text splitters** — loader → splitter ingestion idiom
- **Retrievers + LangChain RAG idiom** — `as_retriever`, `create_stuff_documents_chain`, `create_retrieval_chain`, history-aware retrieval
- **LangSmith tracing** — run-level observability for debugging + optimization
- **LlamaIndex** — connectors, indices, and router query engines
- **LangGraph** — explicit state-machine runtime for production agent workflows

### Practice Assignments (8)

1. **Output Parsers** — Build a typed extractor for support tickets (priority, issue_type, sentiment) with `PydanticOutputParser` and retry/fix logic.
2. **LangChain RAG** — Build a retriever + retrieval chain on TechStore docs and compare answers for `k=2` vs `k=6`.
3. **LangChain Agent** — Add a fourth tool to the Chapter 16 agent and run a query that forces all four tools.
4. **LangGraph Workflow** — Add a human-approval interrupt before one tool and demonstrate pause/resume.
5. **LlamaIndex Query Engine** — Add a third index (e.g., product catalog) and route across all three.
6. **Same RAG, Both Frameworks** — Implement the same question flow in LangChain and LlamaIndex; compare code size, flexibility, and clarity.
7. **LangSmith Deep Dive** — Find your slowest/most expensive run and improve one prompt or tool description; show before/after trace snapshots.
8. **Twitter Follows Tool** — Add a tool that returns mock followed accounts and integrate it into an agentic query flow.

---

**Next up: Section 7** — multi-agent coordination, planning loops, and long-horizon execution patterns.